# 04 - Resolve and download public ENA assemblies

ENA exposes public `SEQUENCE_ASSEMBLY` analysis records for the resolved isolates. Downloading assembled FASTA files is much smaller and simpler than retrieving approximately 40.94 GB of raw FASTQ reads and assembling them locally.

## 1. Configuration

The metadata lookup is cached. The download switch defaults to `False`. Begin with four assemblies and review the estimate before increasing `DOWNLOAD_LIMIT`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from disinfectant_tolerance.ena import https_urls, load_cache, query_assembly_analyses, save_cache

RUN_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_run_retrieval_plan.csv"
ASSEMBLY_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_assembly_retrieval_plan.csv"
ASSEMBLY_SMOKE_PLAN_PATH = PROJECT_ROOT / "data" / "manifests" / "ena_assembly_smoke_plan.csv"
SPLITS_PATH = PROJECT_ROOT / "data" / "processed" / "isolate_splits.csv"
CACHE_PATH = PROJECT_ROOT / "data" / "raw" / "ena_metadata" / "assembly_analysis_cache.json"
DOWNLOAD_LIMIT = 4
MINIMUM_FREE_GB = 15.0
DOWNLOAD_ASSEMBLIES = False

print("Project root:", PROJECT_ROOT)

## 2. Resolve ENA assembly analyses

This queries metadata only. One isolate-level row is retained for each downloadable assembly.

In [ ]:
run_plan = pd.read_csv(RUN_PLAN_PATH, dtype={"label": str}).fillna("")
cache = load_cache(CACHE_PATH)
assembly_rows = []

for position, row in run_plan.iterrows():
    sample_accession = row["sample_accession"]
    if sample_accession not in cache:
        cache[sample_accession] = query_assembly_analyses(sample_accession)
    analyses = cache[sample_accession]
    if analyses:
        analysis = analyses[0]
        urls = https_urls(analysis.get("submitted_ftp", ""))
        if urls:
            assembly_rows.append({
                "isolate_id": row["isolate_id"],
                "label": row["label"],
                "group": row["group"],
                "sample_accession": sample_accession,
                "analysis_accession": analysis.get("analysis_accession", ""),
                "assembly_url": urls[0],
                "submitted_bytes": int(analysis.get("submitted_bytes", 0) or 0),
                "assembly_type": analysis.get("assembly_type", ""),
            })
    if (position + 1) % 20 == 0 or position + 1 == len(run_plan):
        save_cache(CACHE_PATH, cache)
        print(f"Checked {position + 1}/{len(run_plan)} isolates")

assembly_plan = pd.DataFrame(assembly_rows).drop_duplicates("isolate_id")
splits = pd.read_csv(SPLITS_PATH, dtype={"label": str})[["isolate_id", "split"]]
assembly_plan = assembly_plan.merge(splits, on="isolate_id", how="left", validate="one_to_one")
assembly_plan.to_csv(ASSEMBLY_PLAN_PATH, index=False)

smoke_plan = (
    assembly_plan[assembly_plan["split"].isin(["train", "test"])]
    .sort_values("submitted_bytes")
    .groupby(["split", "label"], as_index=False, group_keys=False)
    .head(1)
    .sort_values(["split", "label"])
    .reset_index(drop=True)
)
smoke_plan.to_csv(ASSEMBLY_SMOKE_PLAN_PATH, index=False)

## 3. Review the compact retrieval plan

The decompressed-output estimate uses a conservative five-times multiplier. The downloader preserves at least 15 GB of free local disk space.

In [ ]:
compressed_gb = assembly_plan["submitted_bytes"].sum() / 1_000_000_000
estimated_output_gb = compressed_gb * 5
print("ENA run records:", len(run_plan))
print("Public assembly records:", len(assembly_plan))
print("Assembly coverage: %.1f%%" % (100 * len(assembly_plan) / len(run_plan)))
print("Compressed FASTA total: %.2f GB" % compressed_gb)
print("Conservative decompressed estimate: %.2f GB" % estimated_output_gb)
print("Wrote:", ASSEMBLY_PLAN_PATH.relative_to(PROJECT_ROOT))
print("Wrote:", ASSEMBLY_SMOKE_PLAN_PATH.relative_to(PROJECT_ROOT))
print("\nFour-isolate smoke plan:")
smoke_plan[["isolate_id", "label", "group", "split", "analysis_accession", "submitted_bytes"]]

## 4. Download four public assemblies first

Change `DOWNLOAD_ASSEMBLIES` to `True` after reviewing the estimate. Increase `DOWNLOAD_LIMIT` only when the small download behaves as expected.

In [ ]:
download_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "download_ena_assemblies.py"),
    "--plan",
    str(ASSEMBLY_SMOKE_PLAN_PATH),
    "--output-dir",
    str(PROJECT_ROOT / "data" / "raw" / "assemblies"),
    "--limit-isolates",
    str(DOWNLOAD_LIMIT),
    "--minimum-free-gb",
    str(MINIMUM_FREE_GB),
]
print(" ".join(download_command))
if DOWNLOAD_ASSEMBLIES:
    import subprocess
    subprocess.run(download_command, cwd=PROJECT_ROOT, check=True)

## 5. Next step

Open `05_review_assemblies_and_run_smoke_baseline.ipynb` after downloading at least a small balanced subset. The FASTQ/SPAdes workflow remains available as a fallback for isolates lacking public assemblies, but it is not needed for this cohort.